# GPT-5-разметка manual-check выборки

Этот ноутбук повторно размечает через GPT-5 только manual-check примеры, которые были отобраны для ручной проверки.

## Зачем нужен ноутбук

На предыдущем этапе из GPT-5-размеченного train set было выбрано случайное подмножество отзывов для ручной проверки:

```text
data/labeled/wb_feedbacks_manual_check_random_gpt5/
manual_check_random_by_class.csv
```

Этот ноутбук берет эти же отзывы и прогоняет их через GPT-5 с актуальным промптом.  
Цель — сравнить автоматическую GPT-5-разметку с ручной разметкой `correct_labels`.

## Что делает ноутбук

1. Загружает manual-check файл.
2. Берет текст каждого отзыва.
3. Отправляет отзывы в GPT-5 для multi-label классификации.
4. Сохраняет новые labels в колонку `new_labels`.
5. Оставляет поля для сравнения с ручной проверкой: `correct_labels`, `comment`, `is_correct`.

## Используемая модель

Для повторной разметки используется модель:

```text
gpt-5
```

Модель вызывается через OpenAI API.

Ссылка на документацию OpenAI по моделям:

```text
https://platform.openai.com/docs/models
```

## Выходные данные

```text
data/labeled/wb_feedbacks_manual_check_random_gpt5_prompt_test_v5/
manual_check_random_by_class_gpt5_relabelled.csv
```

В этом файле можно сравнить:

```text
old_labels      — исходная разметка
new_labels      — новая GPT-5-разметка
correct_labels  — ручная эталонная разметка
comment         — комментарий к ошибке
is_correct      — результат ручной проверки
```

## Роль в пайплайне

```text
manual-check выборка
        ↓
ChatGpt_markup_gpt5_manual_check_only.ipynb
        ↓
GPT-5-разметка manual-check примеров
        ↓
сравнение с correct_labels
        ↓
golden set для оценки качества модели
```

Важно: этот ноутбук не формирует train set и не обучает классификатор.  
Он нужен для проверки качества промпта и получения файла, где можно сопоставить GPT-5-разметку с ручной эталонной разметкой.

In [53]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [54]:
# В Google Colab раскомментируй установку зависимостей:
# !pip install -q openai pandas tqdm

import os
import json
import time
from pathlib import Path
from typing import Any

import pandas as pd
from tqdm.auto import tqdm

from openai import OpenAI

## 1. Настройки

`MANUAL_CHECK_PATH` должен указывать на старый файл ручной проверки.  
`OUTPUT_DIR` лучше сделать отдельной папкой, чтобы не перезаписать старую ручную проверку.


In [ ]:
# ====== ПУТИ ======

PROJECT_ROOT = Path("/content/drive/MyDrive/MLops_project/project/data").resolve()
LABELED_DIR = PROJECT_ROOT / "labeled"

# Старый файл ручной проверки, на котором ты уже проверял ошибки.
# Если у тебя другое имя папки — поменяй только этот путь.
MANUAL_CHECK_PATH = (
    LABELED_DIR
    / "wb_feedbacks_manual_check_random_v1"
    / "manual_check_random_by_class.csv"
)

# Отдельная папка для быстрого GPT-5 теста на manual check примерах.
OUTPUT_DIR = LABELED_DIR / "wb_feedbacks_manual_check_random_gpt5_prompt_test_v5"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PROGRESS_JSONL_PATH = OUTPUT_DIR / "manual_check_gpt5_progress.jsonl"
OUTPUT_PATH = OUTPUT_DIR / "manual_check_random_by_class_gpt5_relabelled.csv"
ERRORS_OUTPUT_PATH = OUTPUT_DIR / "manual_check_random_by_class_gpt5_errors_only.csv"
SUMMARY_OUTPUT_PATH = OUTPUT_DIR / "manual_check_random_by_class_gpt5_summary.csv"

TEXT_COLUMN = "отзыв"
OLD_LABELS_COLUMN = "old_labels"
NEW_LABELS_COLUMN = "new_labels"
CORRECT_LABELS_COLUMN = "correct_labels"

# Если True — размечаем только строки, где уже заполнен correct_labels.
# Если False — размечаем все строки из manual check файла.
ONLY_ROWS_WITH_CORRECT_LABELS = False

# ====== OPENAI ======

MODEL = "gpt-5"
BATCH_SIZE = 1
SLEEP_SECONDS = 1
MAX_COMPLETION_TOKENS = 1200
MAX_REVIEWS = None  # например 20 для теста; None — все строки manual check

# Ключ лучше хранить в переменных окружения / Colab Secrets.
# При необходимости временно раскомментируй и вставь свой ключ:
os.environ["OPENAI_API_KEY"] = ""

client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("MANUAL_CHECK_PATH:", MANUAL_CHECK_PATH)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("MODEL:", MODEL)
print("BATCH_SIZE:", BATCH_SIZE)
print("MAX_COMPLETION_TOKENS:", MAX_COMPLETION_TOKENS)

PROJECT_ROOT: /content/drive/MyDrive/MLops_project/project/data
MANUAL_CHECK_PATH: /content/drive/MyDrive/MLops_project/project/data/labeled/wb_feedbacks_manual_check_random_v1/manual_check_random_by_class.csv
OUTPUT_DIR: /content/drive/MyDrive/MLops_project/project/data/labeled/wb_feedbacks_manual_check_random_gpt5_prompt_test_v5
MODEL: gpt-5
BATCH_SIZE: 1
MAX_COMPLETION_TOKENS: 1200


In [56]:
ALLOWED_LABELS_MVP = [
    "Проблема с качеством товара",
    "Проблема с размером / посадкой",
    "Несоответствие карточке товара",
    "Проблема с комплектацией / упаковкой",
    "Проблема доставки / получения",
    "Цена / ценность",
    "Проблема с возвратом",
    "Положительный / нейтральный отзыв",
    "Другая проблема",
]

ALLOWED_LABELS_SET = set(ALLOWED_LABELS_MVP)

## 2. Промпт

Промпт тот же, что в основной GPT-5 разметке. Модель получает только текст отзыва, без старых labels, чтобы разметить заново по смыслу.


In [57]:
SYSTEM_PROMPT = """
Ты ассистент для разметки отзывов покупателей о товарах на маркетплейсе.

Нужно выбрать labels для каждого отзыва.

Это multi-label классификация:

* один отзыв может иметь несколько labels;
* выбирай несколько labels только если в отзыве явно есть несколько разных проблем;
* не выбирай label только по отдельному слову-триггеру;
* размечай по смыслу жалобы, а не по формальным словам;
* если конкретной жалобы нет, выбирай только "Положительный / нейтральный отзыв";
* если отзыв содержит конкретную проблему, НЕ выбирай "Положительный / нейтральный отзыв";
* если жалоба есть, но она не подходит ни под один основной класс, выбирай "Другая проблема";
* если уже выбран конкретный проблемный label, НЕ добавляй "Другая проблема";
* не используй классы вне разрешенного списка;
* не добавляй объяснения, evidence, confidence, markdown или любой текст вне JSON.

Разрешенные labels:

1. "Проблема с качеством товара"
   Брак, дефект, поломка, повреждение самого товара, плохой материал, запах, плохой пошив, царапины, сколы, трещины, хлипкость, подделка, плохая сборка, плохое состояние товара, товар не выполняет свою базовую функцию.

Ставь этот label, если:

* товар сломан, разбит, погнут, потерт, поцарапан, треснул, сколот, деформирован;
* товар выглядит как использованный, испорченный, "раздербаненный", "убитый", "покоцанный";
* материал плохой, тонкий, неприятный, хлипкий, колется, рвется, отваливается;
* товар не работает или не выполняет основную функцию: лента не держится, пузыри не получаются, открывашка гнется, пудра выпадает из футляра;
* отзыв коротко говорит, что товар бесполезный, ужасный, деньги на ветер, если из смысла понятно, что товар плохой или не работает.

Важно:

* "Деньги на ветер" само по себе не означает "Цена / ценность"; если причина в том, что товар плохой или не работает, ставь "Проблема с качеством товара".
* "Бесполезная трата денег и времени ожидания" без конкретной цены или доставки чаще относится к плохому качеству товара, а не к цене или доставке.
* Если товар не выполняет базовую функцию, это "Проблема с качеством товара", а не "Несоответствие карточке товара", если нет явного сравнения с карточкой.
* Если материал просто не совпал с заявленным или ожидаемым, это "Несоответствие карточке товара", а не "Проблема с качеством товара".
* Если главный смысл жалобы в том, что материал оказался не тем, который ожидался или был указан в карточке, не добавляй "Проблема с качеством товара" только из-за слова "синтетика", "неприятно" или "не такая ткань".
* Добавляй "Проблема с качеством товара" к несоответствию по материалу только если есть отдельная явная жалоба на качество материала как самостоятельную проблему: материал рвется, колется, вызывает сильный дискомфорт, ткань плохая/хлипкая/непригодная, качество материала плохое или "не дотягивает".
* Не добавляй "Проблема с качеством товара" только потому, что покупатель возвращает товар из-за того, что материал оказался не тем, который был заявлен. Если смысл в том, что "указано хлопок, а пришла синтетика", это "Несоответствие карточке товара".
* Если покупатель прямо пишет "качество могло быть лучше", "качество материала не дотягивает", "качество не соответствует ожиданиям", "качество так себе", это "Проблема с качеством товара".
* "Ожидала хлопковую ткань, а пришла синтетика какая-то, ужасно неприятно на ощупь." → ["Несоответствие карточке товара"]
  Причина: главный смысл — материал не совпал с ожиданием/карточкой; качество отдельно не добавляем.
* "... возврат потому, что 100% синтетика ... хотя указано хлопок ..." → ["Несоответствие карточке товара"]
  Причина: главный смысл — состав не соответствует заявленному; возврат здесь просто действие покупателя, а не отдельная проблема качества или возврата.

2. "Проблема с размером / посадкой"
   Маломерит, большемерит, не подошел размер, плохо сидит, слишком узкое/широкое/короткое/длинное, не подходит по форме, посадке, габаритам или совместимости.

Ставь этот label, если:

* одежда/обувь не подошла по размеру или посадке;
* товар не подходит по форме или габаритам;
* аксессуар/деталь не совместимы с нужным предметом.

3. "Несоответствие карточке товара"
   Товар отличается от фото, описания, характеристик, состава, цвета, количества, модели, варианта, комплектации, заявленного материала, оригинальности или заявленного эффекта.

Ставь этот label, если:

* привезли не тот товар;
* заказали один вариант, а привезли другой;
* заказали один цвет/модель/тип/артикул/название, а пришел другой;
* товар отличается от фото или описания;
* заявлена нержавейка, оригинал, конкретный состав или материал, но покупатель пишет, что это неправда;
* покупатель прямо пишет "подделка", "не оригинал", "обман", если смысл в том, что товар не соответствует заявленному;
* ожидался один материал/состав, а пришел другой: ожидала хлопок, а пришла синтетика; думала будет хлопок, а пришла синтетика; указано хлопок, а фактически 100% синтетика; заявлена кожа, а пришел кожзам;
* покупатель пишет, что материал, состав или ткань не соответствуют заявленному, ожидаемому или указанному в карточке;
* покупатель пишет, что карточка/фото/ракурс вводит в заблуждение: товар показан не той стороной, выгодным ракурсом, без понятного масштаба, из-за чего размер/вид товара воспринимается неправильно;
* покупатель пишет "не вводите покупателей в заблуждение", "показывайте лицевой стороной", "на фото выглядит иначе", если смысл в том, что карточка создала неверное ожидание о товаре.

Важно:

* "Привезли не тот наполнитель: заказывала 'Сенсацию свежести', привезли 'Классик'" → это "Несоответствие карточке товара".
* "Может специально привозят не тот товар" → это "Несоответствие карточке товара".
* Если товар просто плохой, но нет явного отличия от карточки/фото/описания/заказанного варианта, не ставь этот label.
* Если товар не работает как должен, но нет указания на обещания в карточке, это обычно "Проблема с качеством товара".
* Если в отзыве есть противопоставление "ожидала/думала/заказывала/указано/заявлено X, а пришло Y", это "Несоответствие карточке товара".
* Если главный смысл жалобы в том, что материал не тот, который ожидался или был заявлен, ставь "Несоответствие карточке товара".
* Если покупатель пишет "ожидала хлопок, а пришла синтетика", "думала будет хлопок, а это синтетика", "указано хлопок, а пришла синтетика" — основной label "Несоответствие карточке товара".
* Не добавляй "Проблема с качеством товара" только из-за слова "синтетика", если главный смысл в том, что материал не совпал с ожиданием/описанием/карточкой.
* Если покупатель пишет, что материал не соответствует заявленному, ожидаемому или указанному в карточке, по умолчанию ставь только "Несоответствие карточке товара".
* Не добавляй "Проблема с качеством товара" только потому, что покупатель возвращает товар из-за несоответствия материала.
* Добавляй "Проблема с качеством товара" дополнительно только если есть отдельная явная жалоба на качество материала: материал рвется, колется, вызывает сильный дискомфорт, ткань плохая/хлипкая/непригодная, качество материала плохое или "не дотягивает".

4. "Проблема с комплектацией / упаковкой"
   Неполный комплект, не хватает деталей или товара, неправильное/поврежденное вложение, нет инструкции, плохая инструкция, упаковка повреждена, коробка мятая, порванная или плохо упакована.

Ставь этот label, если:

* не хватает товара, детали, аксессуара, инструкции;
* инструкция есть, но с ней есть проблема: она непонятная, слишком мелкая, плохо читается, плохо объясняет использование, не хватает картинок, схем или объяснений;
* жалоба относится к инструкции, руководству, схеме сборки или объяснениям на коробке/в комплекте;
* из упаковки что-то вырезали, достали, украли или оно отсутствует;
* упаковка вскрыта, нарушена, порвана, помята, повреждена;
* коробка в плохом состоянии;
* товар не подходит для подарка из-за состояния коробки;
* этикетки наклеены плохо, оторваны или повреждены;
* товар плохо упакован.

Важно:

* "Коробка в хлам" → это "Проблема с комплектацией / упаковкой".
* "Коробка в плачевном состоянии, для подарка не подошла" → это "Проблема с комплектацией / упаковкой".
* "В ПВЗ обнаружили, что ножниц нет, вырезали их" → это "Проблема с комплектацией / упаковкой".
* Если упаковка нарушена и сам товар из-за этого выглядит поврежденным/раздербаненным, можно дополнительно поставить "Проблема с качеством товара".
* Проблемы с инструкцией относятся к "Проблема с комплектацией / упаковкой", если покупатель жалуется на отсутствие инструкции, непонятную инструкцию, мелкий текст инструкции, отсутствие картинок/схем/объяснений или сложность разобраться по инструкции.
* "Инструкция вообще непонятная, не хватает картинок и объяснений." → ["Проблема с комплектацией / упаковкой"]
* "Инструкция на коробке написана мелко и непонятно, пришлось разбираться долго." → ["Проблема с комплектацией / упаковкой"]

5. "Проблема доставки / получения"
   Долгая доставка, задержка, перенос срока, курьер, пункт выдачи, склад, потеря заказа, отмена заказа, проблема с получением, неправильное вложение на этапе доставки/получения.

Ставь этот label, если:

* доставка задержалась или срок перенесли;
* заказ потеряли или отменили;
* есть жалоба на курьера;
* есть жалоба на ПВЗ;
* есть жалоба на склад;
* проблема обнаружена при получении и покупатель прямо связывает ее со службой доставки, ПВЗ, курьером или складом;
* неправильное вложение связано с доставкой/складом/ПВЗ/курьером.

Важно:

* Не ставь этот label только потому, что в отзыве есть слова "доставка", "привезли", "пришло", "получила".
* "Сам чемодан супер, но доставка, коробка в хлам" → только "Проблема с комплектацией / упаковкой", потому что конкретная жалоба про коробку, а не про срок, курьера, ПВЗ или потерю.
* "Вот такая служба доставки" при отсутствующем товаре/вырезанном вложении → ставь "Проблема доставки / получения", потому что покупатель явно связывает проблему со службой доставки.
* "Доставка быстрая" не является проблемой доставки.

6. "Цена / ценность"
   Дорого, цена завышена, не стоит своих денег, плохое соотношение цены и качества, качество не соответствует цене, ожидал больше за эти деньги.

Ставь этот label, если:

* покупатель явно жалуется на высокую цену;
* покупатель пишет, что товар не стоит своих денег;
* покупатель сравнивает слабое качество с высокой ценой;
* есть смысл "для такой цены товар слишком плохой".

Важно:

* Не ставь этот label при любой фразе про деньги.
* "Деньги на ветер" часто означает, что товар плохой/не работает, а не что цена завышена.
* "Бесполезная трата денег" без жалобы на цену → обычно "Проблема с качеством товара".
* "Переплатили" из-за неправильного товара или платного возврата → не "Цена / ценность", если нет жалобы на обычную цену товара.
* "Платный возврат", "сняли деньги за возврат", "возврат почти 500₽" → это "Проблема с возвратом", а не "Цена / ценность".
* "Тонкий пластик за большие деньги. Возврат" → "Цена / ценность"; "Проблема с возвратом" не нужна, потому что слово "возврат" просто обозначает решение вернуть товар.
* Если покупатель пишет, что товар маленький/меньше ожидаемого и "не стоит своих денег", "в магазине дешевле", "намного дешевле" — ставь "Цена / ценность". Если при этом он жалуется, что карточка/фото ввели в заблуждение, дополнительно ставь "Несоответствие карточке товара".
* Если покупатель пишет, что "качество материала не дотягивает до заплаченной суммы", "качество не соответствует цене", "за эти деньги качество плохое" — ставь оба labels: "Проблема с качеством товара" и "Цена / ценность".
* Если есть только "за такую сумму" без явной жалобы на завышенную цену или невыгодность, не обязательно ставить "Цена / ценность"; смотри на основную жалобу.

7. "Проблема с возвратом"
   Жалобы на возврат товара или денег: не приняли возврат, отказали в возврате, сложно оформить возврат, долго возвращают деньги, не пришли деньги, списали деньги за возврат, дорогой возврат, платный возврат, удержание денег при отказе/возврате, потеря процента выкупа.

Ставь этот label, если:

* покупатель жалуется, что должен платить за возврат;
* покупатель жалуется на стоимость возврата;
* покупатель пишет "платный возврат", "возврат платный", "возврат почти 500 рублей";
* списали деньги за возврат;
* удержали деньги при возврате или отказе;
* не вернули деньги;
* возврат не приняли;
* возврат затруднен;
* покупатель говорит, что из-за ошибки продавца, склада, ПВЗ, курьера или доставки он не должен платить возврат;
* покупатель пишет, что из-за неправильного товара, неправильного вложения, поврежденного товара или плохой упаковки он теряет деньги;
* покупатель пишет про процент выкупа, снятие процента выкупа, платный отказ или удержание денег из-за неправильного товара.

Очень важно:

* Если покупатель жалуется, что из-за неправильного товара, ошибки продавца, ошибки доставки, поврежденного товара или плохой упаковки он должен платить возврат, теряет деньги, теряет процент выкупа или с него что-то списывают — обязательно ставь "Проблема с возвратом".
* Это правило работает даже если основная проблема относится к другому классу.
* Если привезли не тот товар и из-за этого покупатель пишет про возврат, процент выкупа, списание денег или платный отказ — ставь сразу два labels: "Несоответствие карточке товара" и "Проблема с возвратом".
* Если повреждена коробка/упаковка и покупатель жалуется на платный возврат — ставь сразу два labels: "Проблема с комплектацией / упаковкой" и "Проблема с возвратом".
* Если товар сломан/разбит/поврежден и покупатель жалуется на списание денег за возврат — ставь сразу два labels: "Проблема с качеством товара" и "Проблема с возвратом".

Не ставь этот label, если:

* в отзыве просто есть слово "возврат";
* покупатель просто пишет "вернула", "отправила обратно", "оформила возврат";
* "возврат" используется только как действие покупателя, без жалобы на платность, деньги, отказ, удержание, компенсацию или процент выкупа;
* покупатель пишет, что товар "подлежит возврату", "буду возвращать", "оформила возврат", но не жалуется на сам процесс возврата, платность, отказ, удержание денег или фактическое списание;
* покупатель только предполагает возможное списание денег: "надеюсь 100 р. не снимут", "надеюсь не удержат", без факта списания или жалобы на платный возврат.

Примеры:

* "Привезли не тот товар, чтобы снять процент выкупа" → ["Несоответствие карточке товара", "Проблема с возвратом"]
* "Привезли не тот наполнитель. Почему я должна платить возврат из-за вашей ошибки" → ["Несоответствие карточке товара", "Проблема с возвратом"]
* "Коробка в плохом состоянии, а возврат почти 500 рублей" → ["Проблема с комплектацией / упаковкой", "Проблема с возвратом"]
* "Сняли деньги за возврат поврежденного товара" → ["Проблема с качеством товара", "Проблема с возвратом"]
* "Тонкий пластик за большие деньги. Возврат" → ["Цена / ценность"]
  Причина: здесь слово "возврат" просто обозначает решение вернуть товар, нет жалобы на сам возврат.

8. "Положительный / нейтральный отзыв"
   Положительный, нейтральный или информационный отзыв без явной жалобы.

Ставь этот label, если:

* отзыв положительный;
* отзыв нейтральный;
* покупатель просто сообщает факт без проблемы;
* товар понравился;
* все соответствует;
* отзыв без конкретной жалобы.

Важно:

* Если отзыв содержит позитивную часть, но есть конкретная проблема, не выбирай этот label.
* "Сам чемодан супер, но коробка в хлам" → не ставь "Положительный / нейтральный отзыв".

9. "Другая проблема"
   Есть негатив или жалоба, но она не подходит к классам выше.

Ставь этот label только если:

* есть негативная жалоба;
* невозможно уверенно отнести ее к качеству, размеру, несоответствию, упаковке, доставке, цене или возврату.

Важно:

* Не ставь "Другая проблема", если можно выбрать конкретный label.
* Не ставь "Другая проблема" вместе с конкретным проблемным label.
* "Ожидала лучше" без конкретики → "Другая проблема".
* "Да уж… лента не держится" → не "Другая проблема", а "Проблема с качеством товара".
* "Ерунда. Пузыри не получаются" → не "Другая проблема", а "Проблема с качеством товара".

Главные правила разграничения:

1. Не размечай по словам-триггерам.

* "Доставка" не всегда означает "Проблема доставки / получения".
* "Возврат" не всегда означает "Проблема с возвратом".
* "Деньги" не всегда означает "Цена / ценность".
* "Привезли" не всегда означает доставку.
* "Отправила обратно" не всегда означает проблему с возвратом.
* "Синтетика" не всегда означает "Проблема с качеством товара": если смысл в несоответствии заявленному/ожидаемому составу, это "Несоответствие карточке товара".

2. Качество vs несоответствие карточке.

* Товар плохой, сломан, хлипкий, не работает, не выполняет функцию → "Проблема с качеством товара".
* Товар отличается от фото, описания, заказанного варианта, заявленного материала, модели, цвета, состава или оригинальности → "Несоответствие карточке товара".
* Если есть и плохое качество, и явное несоответствие заявленному, ставь оба labels.
* Если материал отличается от ожидаемого/заявленного, по умолчанию ставь "Несоответствие карточке товара".
* Если материал отличается от ожидаемого/заявленного, но жалоба на качество материала является отдельной проблемой и причиной невозможности использовать/носить/оставить товар, добавь "Проблема с качеством товара".

3. Качество vs цена.

* "Товар плохой / бесполезный / не работает" → "Проблема с качеством товара".
* "Товар слишком плохой именно за такую цену / дорого / не стоит своих денег" → "Цена / ценность".
* Если при этом прямо названа проблема качества: "качество материала не дотягивает", "качество плохое", "качество могло быть лучше" — дополнительно ставь "Проблема с качеством товара".
* "Деньги на ветер" без жалобы на цену → обычно "Проблема с качеством товара".

4. Возврат.

* Ставь "Проблема с возвратом" только при жалобе на процесс возврата, деньги за возврат, платность возврата, удержание денег, отказ, компенсацию или процент выкупа.
* Не ставь этот label, если покупатель просто написал "возврат", "вернула", "отправила обратно".

5. Доставка.

* Ставь "Проблема доставки / получения" только при явной проблеме с доставкой, ПВЗ, курьером, складом, сроками, получением или неправильным вложением на этапе получения.
* Не ставь доставку только потому, что товар "пришел" или "привезли".

6. Упаковка и инструкция.

* Поврежденная коробка, мятая коробка, плохая упаковка, нарушенная упаковка, товар не подходит для подарка из-за коробки → "Проблема с комплектацией / упаковкой".
* Отсутствует товар/деталь/вложение → "Проблема с комплектацией / упаковкой".
* Отсутствует инструкция или инструкция есть, но она плохая/непонятная/мелкая/без картинок/без нормальных объяснений → "Проблема с комплектацией / упаковкой".
* Если покупатель связывает отсутствующее/неправильное вложение с ПВЗ, курьером, складом или службой доставки, добавь "Проблема доставки / получения".

Важные спорные случаи:

* "Сам чемодан супер!!! Но доставка 🤦🏼‍♀️ коробка в хлам …" → ["Проблема с комплектацией / упаковкой"]
  Причина: слово "доставка" здесь контекст, конкретная жалоба — коробка в плохом состоянии.

* "Ребят, ну явная подделка. Этикетки как попало наклеена, коробочка вся покоцанная. Раньше покупала эту пудру, было все нормально. Сейчас я в шоке, мел цветной. Пудра сама прям выпадает из футляра" → ["Проблема с качеством товара", "Проблема с комплектацией / упаковкой", "Несоответствие карточке товара"]
  Причина: плохое качество товара, поврежденная/плохая упаковка и явное подозрение на подделку.

* "В ПВЗ обнаружили, что ножниц то нет, вырезали их. Вот такая служба доставки" → ["Проблема доставки / получения", "Проблема с комплектацией / упаковкой"]
  Причина: отсутствует товар/вложение и покупатель связывает проблему со службой доставки/ПВЗ.

* "Хочу дополнить, может специально привозят не тот товар, чтоб снять процент выкупа" → ["Несоответствие карточке товара", "Проблема с возвратом"]
  Причина: речь о неправильном товаре и удержании/проценте выкупа.

* "Да уж… соплями реальней оклеить, чем этой лентой. Не держится нифига" → ["Проблема с качеством товара"]
  Причина: товар не выполняет базовую функцию.

* "Ужас! Бесполезная трата денег и времени ожидания!" → ["Проблема с качеством товара"]
  Причина: общий смысл — товар бесполезный/плохой; нет конкретной жалобы на цену или доставку.

* "Привезли не тот наполнитель. Заказывала 'Сенсацию свежести', привезли 'Классик'. Почему я из за вашей ошибки должна платить возврат. И в итоге стоимость наполнителя переплатили." → ["Несоответствие карточке товара", "Проблема с возвратом"]
  Причина: привезли другой вариант товара, плюс жалоба на оплату возврата; "Цена / ценность" лишняя, потому что речь не о завышенной цене, а о лишних расходах из-за ошибки.

* "Тонкий пластик за большие деньги. Возврат" → ["Цена / ценность"]
  Причина: основной смысл — слабый материал за высокую цену; "Возврат" здесь просто действие, а не жалоба на возврат; "Проблема с качеством товара" спорна и без явного дефекта не ставится.

* "Коробка в плачевном состоянии. Для подарка бы не подошла, а возврат почти 500₽" → ["Проблема с комплектацией / упаковкой", "Проблема с возвратом"]
  Причина: коробка повреждена, плюс есть жалоба на дорогой возврат.

* "Ерунда. Пузыри не получаются. Деньги на ветер" → ["Проблема с качеством товара"]
  Причина: товар не выполняет базовую функцию; "деньги на ветер" здесь не про цену.

* "Да, мне тоже привозили этих раздербаненных петухов. Отправила обратно. Просила курьера отметить, что неправильное вложение. Упаковка нарушена. Работники склада халатны." → ["Проблема с комплектацией / упаковкой", "Проблема доставки / получения", "Проблема с качеством товара"]
  Причина: товар в плохом состоянии, упаковка нарушена, проблема связана со складом/курьером/неправильным вложением.

* "Заказ пришел быстро, но оба флакона разбиты. Не понятно, за что сняли 100 руб. за возврат" → ["Проблема с качеством товара", "Проблема с возвратом"]
  Причина: товар разбит, плюс жалоба на списание денег за возврат.

* "Ткань тонкая, неприятная" → ["Проблема с качеством товара"]

* "На фото ткань плотная, а пришла тонкая" → ["Несоответствие карточке товара"]

* "Маломерит, но качество хорошее" → ["Проблема с размером / посадкой"]

* "Коробка порвана, но товар целый" → ["Проблема с комплектацией / упаковкой"]

* "Доставили поздно" → ["Проблема доставки / получения"]

* "Ожидала лучше" без конкретики → ["Другая проблема"]

* "Инструкция вообще непонятная, не хватает картинок и объяснений." → ["Проблема с комплектацией / упаковкой"]
  Причина: инструкция входит в комплект/оформление товара, и покупатель жалуется на ее качество.

* "Инструкция на коробке написана мелко и непонятно, пришлось разбираться долго." → ["Проблема с комплектацией / упаковкой"]
  Причина: есть проблема с инструкцией/объяснениями на коробке.

* "Ожидала хлопковую ткань, а пришла синтетика какая-то, ужасно неприятно на ощупь." → ["Несоответствие карточке товара"]
  Причина: главный смысл — материал не совпал с ожиданием/карточкой; качество отдельно не добавляем.

* "100% синтетика, хотя указано хлопок." → ["Несоответствие карточке товара"]
  Причина: прямо указано отличие фактического состава от заявленного.

* "... возврат потому, что 100% синтетика ... хотя указано хлопок ..." → ["Несоответствие карточке товара"]
  Причина: есть несоответствие заявленному составу; возврат здесь просто действие покупателя, а не жалоба на качество или процесс возврата.

* "Качество делало бы лучшего, но впринцепе за такую сумму." → ["Проблема с качеством товара"]
  Причина: есть прямая жалоба на качество; фраза "за такую сумму" здесь не является явной жалобой на цену.

* "Качество материала не дотягивает до заплаченной суммы, ожидал большего." → ["Проблема с качеством товара", "Цена / ценность"]
  Причина: есть прямая жалоба на качество материала и одновременно сравнение качества с заплаченной суммой.

Правила из ручной проверки, которые особенно важно соблюдать:

* Если в отзыве сказано, что привезли не тот товар, другой вариант, другой артикул, другой цвет, другой наполнитель или другой объем — ставь "Несоответствие карточке товара".
* Если вместе с неправильным товаром есть жалоба на платный возврат, процент выкупа, списание денег или платный отказ — дополнительно ставь "Проблема с возвратом".
* Если в ПВЗ обнаружили, что части товара нет, товар вырезали, украли, вскрыли или отсутствует вложение — ставь "Проблема с комплектацией / упаковкой". Если покупатель явно связывает это с ПВЗ/доставкой/службой доставки, дополнительно ставь "Проблема доставки / получения".
* Если товар разбит, сломан, поврежден, протек, треснул или пришел в непригодном состоянии — ставь "Проблема с качеством товара". Если при этом списали деньги за возврат или возврат платный — дополнительно ставь "Проблема с возвратом".
* Если жалоба только на плохую коробку, мятую упаковку или невозможность подарить из-за состояния упаковки — ставь "Проблема с комплектацией / упаковкой", но не ставь доставку без явной жалобы на срок, курьера, ПВЗ, склад или получение.
* Если покупатель пишет, что товар тонкий/хлипкий/плохой "за большие деньги" или "не стоит своих денег" — ставь "Цена / ценность". Не добавляй "Проблема с возвратом", если возврат упомянут только как решение вернуть товар без жалобы на деньги/отказ/платность возврата.
* Если покупатель жалуется на инструкцию, руководство, схему, объяснения на коробке, мелкий текст инструкции или отсутствие картинок/схем — ставь "Проблема с комплектацией / упаковкой".
* Если покупатель пишет, что ожидался/заявлен один материал, а пришел другой — ставь "Несоответствие карточке товара".
* Если материал не соответствует заявленному/ожидаемому, ставь "Несоответствие карточке товара". Сам факт возврата из-за несоответствия материала не добавляет "Проблема с качеством товара".
* Добавляй "Проблема с качеством товара" к несоответствию материала только при отдельной явной жалобе на качество материала: рвется, колется, сильный дискомфорт, плохая/хлипкая/непригодная ткань, качество материала не дотягивает.

Финальная самопроверка перед ответом:

* Не поставлен ли label только из-за слова-триггера?
* Не поставлена ли доставка только из-за слова "доставка", "привезли", "пришло"?
* Не поставлен ли возврат только из-за слова "возврат", "вернула", "отправила обратно"?
* Если в отзыве есть "платить возврат", "платный возврат", "дорогой возврат", "возврат почти ... рублей", "сняли деньги за возврат", "удержали деньги", "процент выкупа", "снять процент выкупа", "из-за вашей ошибки должна платить" — добавлена ли "Проблема с возвратом"?
* Если нет — добавь "Проблема с возвратом".
* Не поставлена ли цена только из-за слов "деньги", "деньги на ветер", "бесполезная трата денег"?
* Не перепутано ли плохое качество с несоответствием карточке?
* Если в отзыве есть "ожидала/думала/указано/заявлено X, а пришло Y", не перепутано ли это с качеством? Обычно это "Несоответствие карточке товара".
* Если жалоба на материал связана только с тем, что материал не совпал с ожиданием/описанием/карточкой, не добавлена ли ошибочно "Проблема с качеством товара"?
* Если материал не совпал с заявленным и из-за этого покупатель возвращает товар, не добавлена ли ошибочно "Проблема с качеством товара"? Сам возврат из-за несоответствия материала не является жалобой на качество.
* Если есть жалоба на инструкцию/руководство/схему/мелкий текст инструкции/нет картинок/непонятные объяснения, добавлена ли "Проблема с комплектацией / упаковкой"?
* Не выбран ли "Положительный / нейтральный отзыв" вместе с проблемой?
* Не выбран ли "Другая проблема" вместе с конкретным проблемным label?
* Если в отзыве есть "качество могло быть лучше", "качество не дотягивает", "качество материала не дотягивает", добавлена ли "Проблема с качеством товара"?
* Если материал не совпал с заявленным, не добавлена ли ошибочно "Проблема с качеством товара" только из-за возврата? Если нет отдельной жалобы на качество материала, оставь только "Несоответствие карточке товара".
* Если качество прямо сравнивается с ценой: "качество не соответствует цене", "качество материала не дотягивает до суммы", добавлены ли оба labels: "Проблема с качеством товара" и "Цена / ценность"?

Верни только JSON строго такого вида:
{
"items": [
{
"id": "тот же id",
"labels": ["один или несколько разрешенных labels"]
}
]
}
"""


In [58]:
RESPONSE_FORMAT = {
    "type": "json_schema",
    "json_schema": {
        "name": "marketplace_review_manual_check_relabeling_mvp",
        "strict": True,
        "schema": {
            "type": "object",
            "additionalProperties": False,
            "properties": {
                "items": {
                    "type": "array",
                    "items": {
                        "type": "object",
                        "additionalProperties": False,
                        "properties": {
                            "id": {"type": "string"},
                            "labels": {
                                "type": "array",
                                "items": {
                                    "type": "string",
                                    "enum": ALLOWED_LABELS_MVP,
                                },
                            },
                        },
                        "required": ["id", "labels"],
                    },
                }
            },
            "required": ["items"],
        },
    },
}

## 3. Вспомогательные функции

In [59]:
def read_csv_safely(path: Path) -> pd.DataFrame:
    encodings = ["utf-8", "utf-8-sig", "cp1251"]
    last_error = None

    for encoding in encodings:
        try:
            return pd.read_csv(path, encoding=encoding)
        except Exception as e:
            last_error = e

    raise RuntimeError(f"Не удалось прочитать файл {path}. Последняя ошибка: {last_error}")


def labels_to_list(labels: Any) -> list[str]:
    if isinstance(labels, list):
        raw = labels
    elif pd.isna(labels):
        raw = []
    elif isinstance(labels, str):
        value = labels.strip()
        if not value:
            raw = []
        else:
            try:
                parsed = json.loads(value)
                raw = parsed if isinstance(parsed, list) else []
            except Exception:
                if " | " in value:
                    raw = [x.strip() for x in value.split(" | ")]
                elif ";" in value:
                    raw = [x.strip() for x in value.split(";")]
                elif "," in value and value not in ALLOWED_LABELS_SET:
                    raw = [x.strip() for x in value.split(",")]
                elif value in ALLOWED_LABELS_SET:
                    raw = [value]
                else:
                    raw = []
    else:
        raw = []

    result = []
    for label in raw:
        label = str(label).strip().strip('"').strip("'")
        if label in ALLOWED_LABELS_SET and label not in result:
            result.append(label)

    return result


def normalize_labels(labels: Any) -> list[str]:
    result = labels_to_list(labels)

    if not result:
        return []

    neutral_label = "Положительный / нейтральный отзыв"

    if neutral_label in result and len(result) > 1:
        result = [label for label in result if label != neutral_label]

    return result


def labels_to_json(labels: Any) -> str:
    return json.dumps(normalize_labels(labels), ensure_ascii=False)


def labels_to_str(labels: Any) -> str:
    return " | ".join(normalize_labels(labels))


def labels_equal(a: Any, b: Any) -> bool | None:
    a_list = normalize_labels(a)
    b_list = normalize_labels(b)

    if not b_list:
        return None

    return set(a_list) == set(b_list)


def is_filled(value: Any) -> bool:
    if pd.isna(value):
        return False
    if isinstance(value, str) and not value.strip():
        return False
    return True


def load_done_ids(progress_path: Path) -> set[str]:
    if not progress_path.exists():
        return set()

    done = set()
    with progress_path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
                done.add(str(obj["manual_check_id"]))
            except Exception:
                continue

    return done


def append_jsonl(path: Path, rows: list[dict[str, Any]]) -> None:
    with path.open("a", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

def make_batches(data: pd.DataFrame, batch_size: int):
    for start in range(0, len(data), batch_size):
        yield data.iloc[start:start + batch_size]

## 4. Загружаем manual check файл

In [60]:
if not MANUAL_CHECK_PATH.exists():
    raise FileNotFoundError(
        f"Не найден файл ручной проверки: {MANUAL_CHECK_PATH}\n"
        "Проверь MANUAL_CHECK_PATH."
    )

manual_df = read_csv_safely(MANUAL_CHECK_PATH).copy()

if TEXT_COLUMN not in manual_df.columns:
    raise ValueError(f"В файле нет колонки {TEXT_COLUMN!r}. Колонки: {list(manual_df.columns)}")

if "manual_check_id" not in manual_df.columns:
    manual_df["manual_check_id"] = [f"manual_check__{i}" for i in range(len(manual_df))]

# Старые labels берем из old_labels, если уже есть; иначе из labels.
if OLD_LABELS_COLUMN not in manual_df.columns:
    if "labels" in manual_df.columns:
        manual_df[OLD_LABELS_COLUMN] = manual_df["labels"]
    else:
        manual_df[OLD_LABELS_COLUMN] = "[]"

# correct_labels оставляем как ручной эталон.
if CORRECT_LABELS_COLUMN not in manual_df.columns:
    manual_df[CORRECT_LABELS_COLUMN] = ""

# Убираем пустые отзывы.
manual_df = manual_df[manual_df[TEXT_COLUMN].notna()].copy()
manual_df[TEXT_COLUMN] = manual_df[TEXT_COLUMN].astype(str).str.strip()
manual_df = manual_df[manual_df[TEXT_COLUMN] != ""].copy()

if ONLY_ROWS_WITH_CORRECT_LABELS:
    before = len(manual_df)
    manual_df = manual_df[manual_df[CORRECT_LABELS_COLUMN].apply(is_filled)].copy()
    print(f"Оставлены только строки с correct_labels: {len(manual_df)} из {before}")

if MAX_REVIEWS is not None:
    manual_df = manual_df.head(MAX_REVIEWS).copy()

manual_df["manual_check_id"] = manual_df["manual_check_id"].astype(str)
manual_df = manual_df.drop_duplicates(subset=["manual_check_id"], keep="first").reset_index(drop=True)

print("Строк для разметки:", len(manual_df))
print("Строк с заполненным correct_labels:", int(manual_df[CORRECT_LABELS_COLUMN].apply(is_filled).sum()))
manual_df.head()

Строк для разметки: 150
Строк с заполненным correct_labels: 0


,manual_check_id,class_for_check,source_type,отзыв,labels,labels_str,is_correct,correct_labels,comment,source_class_file,old_labels
0,проблема_доставки_получения__13,Проблема доставки / получения,synthetic,"Перенесли доставку на неделю, и к тому же това...","[""Проблема доставки / получения"", ""Проблема с ...",Проблема доставки / получения | Проблема с ком...,NaN,NaN,NaN,labeled/wb_feedbacks_by_class_with_synthetic/п...,"[""Проблема доставки / получения"", ""Проблема с ..."
1,проблема_с_размером_посадкой__8,Проблема с размером / посадкой,real,"Свитер большемерит. На рост ребенка 116, р-о 1...","[""Проблема с размером / посадкой"", ""Проблема с...",Проблема с размером / посадкой | Проблема с ка...,NaN,NaN,NaN,labeled/wb_feedbacks_by_class_with_synthetic/п...,"[""Проблема с размером / посадкой"", ""Проблема с..."
2,положительный_нейтральный_отзыв__8,Положительный / нейтральный отзыв,real,"Отличная куртка, размер в размер. Дочь довольн...","[""Положительный / нейтральный отзыв""]",Положительный / нейтральный отзыв,NaN,NaN,NaN,labeled/wb_feedbacks_by_class_with_synthetic/п...,"[""Положительный / нейтральный отзыв""]"
3,проблема_доставки_получения__18,Проблема доставки / получения,synthetic,"Доставка заняла больше месяца, ожидал намного ...","[""Проблема доставки / получения""]",Проблема доставки / получения,NaN,NaN,NaN,labeled/wb_feedbacks_by_class_with_synthetic/п...,"[""Проблема доставки / получения""]"
4,проблема_доставки_получения__16,Проблема доставки / получения,synthetic,"На пункте выдачи не было моего заказа, сказали...","[""Проблема доставки / получения""]",Проблема доставки / получения,NaN,NaN,NaN,labeled/wb_feedbacks_by_class_with_synthetic/п...,"[""Проблема доставки / получения""]"


## 5. Функции GPT-5 разметки

In [61]:
def build_user_prompt(batch_df: pd.DataFrame) -> str:
    reviews = []

    for _, row in batch_df.iterrows():
        reviews.append({
            "id": str(row["manual_check_id"]),
            "text": str(row[TEXT_COLUMN]),
        })

    return (
        "Ниже список отзывов из manual check файла. "
        "На входе только id и текст отзыва. "
        "Старые labels и correct_labels специально не передаются: разметь отзыв заново по смыслу. "
        "Верни JSON строго по схеме. "
        "Сохрани id каждого отзыва без изменений.\n\n"
        + json.dumps({"reviews": reviews}, ensure_ascii=False, indent=2)
    )


def validate_item(item: dict[str, Any]) -> dict[str, Any]:
    labels = item.get("labels", [])

    if not isinstance(labels, list):
        labels = ["Другая проблема"]

    labels = [str(label).strip() for label in labels]
    labels = [label for label in labels if label in ALLOWED_LABELS_SET]

    if not labels:
        labels = ["Другая проблема"]

    neutral_label = "Положительный / нейтральный отзыв"

    if neutral_label in labels and len(labels) > 1:
        labels = [label for label in labels if label != neutral_label]

    deduped = []
    for label in labels:
        if label not in deduped:
            deduped.append(label)

    return {
        "id": str(item.get("id", "")),
        "labels": deduped,
    }


def label_batch(batch_df: pd.DataFrame, max_retries: int = 3) -> list[dict[str, Any]]:
    user_prompt = build_user_prompt(batch_df)
    last_error = None

    for attempt in range(1, max_retries + 1):
        try:
            print(
                f"Отправляю батч в OpenAI API. "
                f"Попытка {attempt}/{max_retries}. "
                f"Размер батча: {len(batch_df)}. "
                f"MODEL={MODEL}. "
                f"MAX_COMPLETION_TOKENS={MAX_COMPLETION_TOKENS}"
            )

            response = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user_prompt},
                ],
                response_format=RESPONSE_FORMAT,
                timeout=300,
                max_completion_tokens=MAX_COMPLETION_TOKENS,
            )

            choice = response.choices[0]

            if choice.finish_reason == "length":
                raise RuntimeError(
                    "Ответ модели был обрезан. "
                    f"finish_reason={choice.finish_reason}. "
                    "Увеличь MAX_COMPLETION_TOKENS или уменьши BATCH_SIZE."
                )

            content = choice.message.content
            parsed = json.loads(content)
            items = parsed.get("items", [])

            if not isinstance(items, list):
                raise ValueError("Поле items не является списком")

            validated = [validate_item(item) for item in items]

            expected_ids = set(batch_df["manual_check_id"].astype(str))
            returned_ids = set(item["id"] for item in validated)

            missing_ids = expected_ids - returned_ids
            extra_ids = returned_ids - expected_ids

            if missing_ids:
                raise ValueError(f"Модель не вернула id: {sorted(missing_ids)}")

            if extra_ids:
                print(f"Предупреждение: модель вернула лишние id: {sorted(extra_ids)}")

            return [item for item in validated if item["id"] in expected_ids]

        except Exception as e:
            last_error = e
            print(f"Ошибка на попытке {attempt}/{max_retries}: {e}")

            if attempt < max_retries:
                wait_seconds = 20
                print(f"Ждем {wait_seconds} сек. и пробуем снова...")
                time.sleep(wait_seconds)

    raise RuntimeError(f"Батч не удалось разметить после {max_retries} попыток: {last_error}")

## 6. Быстрый тест на первых строках

Запусти эту ячейку перед полной разметкой, чтобы проверить, что API-вызов работает.


In [62]:
test_batch = manual_df.head(min(BATCH_SIZE, len(manual_df))).copy()
test_result = label_batch(test_batch)

test_result_df = pd.DataFrame(test_result).rename(columns={"labels": NEW_LABELS_COLUMN})
test_result_df = test_batch[["manual_check_id", TEXT_COLUMN, OLD_LABELS_COLUMN, CORRECT_LABELS_COLUMN]].merge(
    test_result_df,
    left_on="manual_check_id",
    right_on="id",
    how="left",
)
test_result_df[["manual_check_id", TEXT_COLUMN, OLD_LABELS_COLUMN, NEW_LABELS_COLUMN, CORRECT_LABELS_COLUMN]]

Отправляю батч в OpenAI API. Попытка 1/3. Размер батча: 1. MODEL=gpt-5. MAX_COMPLETION_TOKENS=1200


,manual_check_id,отзыв,old_labels,new_labels,correct_labels
0,проблема_доставки_получения__13,"Перенесли доставку на неделю, и к тому же това...","[""Проблема доставки / получения"", ""Проблема с ...","[Проблема доставки / получения, Проблема с ком...",NaN


## 7. Полная разметка manual check файла

In [63]:
done_ids = load_done_ids(PROGRESS_JSONL_PATH)
print("Уже размечено:", len(done_ids))

todo_df = manual_df[~manual_df["manual_check_id"].astype(str).isin(done_ids)].copy()
print("Осталось разметить:", len(todo_df))

batches = list(make_batches(todo_df, BATCH_SIZE))

for batch_df in tqdm(batches, desc="Разметка manual check батчей"):
    labeled_items = label_batch(batch_df)

    meta_by_id = {
        str(row["manual_check_id"]): row.to_dict()
        for _, row in batch_df.iterrows()
    }

    rows_to_save = []

    for item in labeled_items:
        manual_check_id = str(item["id"])
        meta = meta_by_id.get(manual_check_id, {})
        new_labels = item["labels"]

        row_to_save = dict(meta)
        row_to_save["manual_check_id"] = manual_check_id
        row_to_save[NEW_LABELS_COLUMN] = new_labels
        rows_to_save.append(row_to_save)

    append_jsonl(PROGRESS_JSONL_PATH, rows_to_save)
    time.sleep(SLEEP_SECONDS)

print("Готово.")

Уже размечено: 0
Осталось разметить: 150


Разметка manual check батчей:   0%|          | 0/150 [00:00<?, ?it/s]

Отправляю батч в OpenAI API. Попытка 1/3. Размер батча: 1. MODEL=gpt-5. MAX_COMPLETION_TOKENS=1200
Отправляю батч в OpenAI API. Попытка 1/3. Размер батча: 1. MODEL=gpt-5. MAX_COMPLETION_TOKENS=1200
Отправляю батч в OpenAI API. Попытка 1/3. Размер батча: 1. MODEL=gpt-5. MAX_COMPLETION_TOKENS=1200
Отправляю батч в OpenAI API. Попытка 1/3. Размер батча: 1. MODEL=gpt-5. MAX_COMPLETION_TOKENS=1200
Отправляю батч в OpenAI API. Попытка 1/3. Размер батча: 1. MODEL=gpt-5. MAX_COMPLETION_TOKENS=1200
Отправляю батч в OpenAI API. Попытка 1/3. Размер батча: 1. MODEL=gpt-5. MAX_COMPLETION_TOKENS=1200
Отправляю батч в OpenAI API. Попытка 1/3. Размер батча: 1. MODEL=gpt-5. MAX_COMPLETION_TOKENS=1200
Отправляю батч в OpenAI API. Попытка 1/3. Размер батча: 1. MODEL=gpt-5. MAX_COMPLETION_TOKENS=1200
Отправляю батч в OpenAI API. Попытка 1/3. Размер батча: 1. MODEL=gpt-5. MAX_COMPLETION_TOKENS=1200
Отправляю батч в OpenAI API. Попытка 1/3. Размер батча: 1. MODEL=gpt-5. MAX_COMPLETION_TOKENS=1200
Отправляю 

## 8. Собираем итоговый CSV и отчет по ошибкам

In [64]:
def read_progress_jsonl(path: Path) -> pd.DataFrame:
    rows = []

    if not path.exists():
        return pd.DataFrame()

    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))

    return pd.DataFrame(rows)


labeled_df = read_progress_jsonl(PROGRESS_JSONL_PATH)

if labeled_df.empty:
    raise RuntimeError("Файл прогресса пустой. Сначала запусти разметку.")

labeled_df = labeled_df.drop_duplicates(subset=["manual_check_id"], keep="last").reset_index(drop=True)

# Приводим основные колонки к понятному виду.
labeled_df[OLD_LABELS_COLUMN] = labeled_df[OLD_LABELS_COLUMN].apply(labels_to_json)
labeled_df[NEW_LABELS_COLUMN] = labeled_df[NEW_LABELS_COLUMN].apply(labels_to_json)
labeled_df[CORRECT_LABELS_COLUMN] = labeled_df[CORRECT_LABELS_COLUMN].apply(labels_to_json)

labeled_df["old_labels_str"] = labeled_df[OLD_LABELS_COLUMN].apply(labels_to_str)
labeled_df["new_labels_str"] = labeled_df[NEW_LABELS_COLUMN].apply(labels_to_str)
labeled_df["correct_labels_str"] = labeled_df[CORRECT_LABELS_COLUMN].apply(labels_to_str)

labeled_df["labels_changed"] = labeled_df.apply(
    lambda row: set(normalize_labels(row[OLD_LABELS_COLUMN])) != set(normalize_labels(row[NEW_LABELS_COLUMN])),
    axis=1,
)

labeled_df["new_equals_correct"] = labeled_df.apply(
    lambda row: labels_equal(row[NEW_LABELS_COLUMN], row[CORRECT_LABELS_COLUMN]),
    axis=1,
)

# Для совместимости labels = new_labels.
labeled_df["labels"] = labeled_df[NEW_LABELS_COLUMN]
labeled_df["labels_str"] = labeled_df["new_labels_str"]

# Удобный порядок колонок.
priority_cols = [
    "manual_check_id",
    "class_for_check",
    "source_type",
    TEXT_COLUMN,
    OLD_LABELS_COLUMN,
    NEW_LABELS_COLUMN,
    CORRECT_LABELS_COLUMN,
    "old_labels_str",
    "new_labels_str",
    "correct_labels_str",
    "labels_changed",
    "new_equals_correct",
    "comment",
    "source_class_file",
]

other_cols = [col for col in labeled_df.columns if col not in priority_cols]
output_df = labeled_df[[col for col in priority_cols if col in labeled_df.columns] + other_cols].copy()

output_df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")

errors_df = output_df[output_df["new_equals_correct"] == False].copy()
errors_df.to_csv(ERRORS_OUTPUT_PATH, index=False, encoding="utf-8-sig")

summary = {
    "total_rows": len(output_df),
    "rows_with_correct_labels": int(output_df["new_equals_correct"].notna().sum()),
    "labels_changed": int(output_df["labels_changed"].sum()),
    "new_equals_correct_true": int((output_df["new_equals_correct"] == True).sum()),
    "new_equals_correct_false": int((output_df["new_equals_correct"] == False).sum()),
}

summary_df = pd.DataFrame([summary])
summary_df.to_csv(SUMMARY_OUTPUT_PATH, index=False, encoding="utf-8-sig")

print("Сохранено:", OUTPUT_PATH)
print("Ошибки сохранены:", ERRORS_OUTPUT_PATH)
print("Summary сохранен:", SUMMARY_OUTPUT_PATH)
summary_df

Сохранено: /content/drive/MyDrive/MLops_project/project/data/labeled/wb_feedbacks_manual_check_random_gpt5_prompt_test_v5/manual_check_random_by_class_gpt5_relabelled.csv
Ошибки сохранены: /content/drive/MyDrive/MLops_project/project/data/labeled/wb_feedbacks_manual_check_random_gpt5_prompt_test_v5/manual_check_random_by_class_gpt5_errors_only.csv
Summary сохранен: /content/drive/MyDrive/MLops_project/project/data/labeled/wb_feedbacks_manual_check_random_gpt5_prompt_test_v5/manual_check_random_by_class_gpt5_summary.csv


,total_rows,rows_with_correct_labels,labels_changed,new_equals_correct_true,new_equals_correct_false
0,150,0,49,0,0


## 9. Посмотреть оставшиеся ошибки

In [65]:
errors_df[[
    "manual_check_id",
    TEXT_COLUMN,
    "old_labels_str",
    "new_labels_str",
    "correct_labels_str",
    "comment",
]].head(30)

,manual_check_id,отзыв,old_labels_str,new_labels_str,correct_labels_str,comment
